In [1]:
import cv2
import numpy as np

In [2]:
from ultralytics import YOLO

# Load model pretrained (COCO)
model = YOLO("yolov8n.pt")

In [3]:

from sklearn.cluster import KMeans

def detect_traffic_light_color_kmeans(roi, k=4):
    hsv_roi=cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    pixels = hsv_roi.reshape(-1, 3)
    pixels = pixels[pixels[:, 2] > 100]  # V > 50

    if len(pixels) == 0:
        return "UNKNOWN"

    kmeans = KMeans(n_clusters=k, n_init=7)
    kmeans.fit(pixels)

    centers = kmeans.cluster_centers_

    centers = np.array(centers)

    scores = centers[:, 1] * centers[:, 2]  # S * V
    best = centers[np.argmax(scores)]

    h, s, v = best

    # 5. điều kiện phân loại màu (Hue)
    if (h < 10) or (h > 170):
        return "RED"
    elif 50 < h < 100:
        return "GREEN"
    elif 15 < h < 35:
        return "YELLOW"
    else:
        return "UNKNOWN"

In [5]:
# cap = cv2.VideoCapture(r"D:\cdio3\Download.mp4")

# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     results = model(frame)

#     for r in results:
#         for box in r.boxes:
#             cls_id = int(box.cls[0])
#             class_name = model.names[cls_id]

#             if class_name == "traffic light":
#                 x1, y1, x2, y2 = map(int, box.xyxy[0])

#                 # cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
#                 crop = frame[y1:y2, x1:x2]
#                 print(f"crop: {crop.shape}")
#                 color = detect_traffic_light_color_kmeans(crop)

#                 cv2.putText(frame, color, (x1, y1-10),
#                             cv2.FONT_HERSHEY_SIMPLEX, 0.7,
#                             (22,55,67), 2)

#     cv2.imshow("video", frame)

#     if cv2.waitKey(1) == 27:
#         break

# cap.release()
# cv2.destroyAllWindows()

In [6]:
# import cv2

# def get_hsv(event, x, y, flags, param):
#     if event == cv2.EVENT_LBUTTONDOWN:
#         pixel = hsv[y, x]
#         print(f"H: {pixel[0]}, S: {pixel[1]}, V: {pixel[2]}")

# img = cv2.imread(r"D:\cdio3\anhvideo\frame_0293.jpg")
# hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# cv2.imshow("image", img)
# cv2.setMouseCallback("image", get_hsv)

# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [10]:
model = YOLO("yolov8s.pt") 
cap = cv2.VideoCapture(r"D:\cdio3\video_den_do.mp4")
history = {}
count_his = []
count = 0
c_line_x1 = 300
c_line_x2 = 700
c_line_y1 = 200
c_line_y2 = 400
color = "green"
def is_inside_roi(cx, cy):
    return c_line_x1 < cx < c_line_x2 and c_line_y1 < cy < c_line_y2
while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame, persist=True, tracker="bytetrack.yaml")

    for r in results:
        boxes = r.boxes

        for box in boxes:
            if box.id is None:
                continue

            track_id = int(box.id[0])
            cls = int(box.cls[0])

         
            if cls in [2, 3, 5, 7]:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2
                state=is_inside_roi(cx,cy)
                if track_id not in history:
                    history[track_id] = state
                    continue

                prev_sate = history[track_id]
                history[track_id] = state
                
              
                if not prev_sate and state and (track_id not in count_his):
                    if color.lower() == "red":
                        count += 1
                        count_his.append(track_id)
                        print(f"VI PHAM ID: {track_id}")

                
                cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
                cv2.putText(frame, f"ID {track_id}", (x1, y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)
            if cls == 9:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    crop = frame[y1:y2, x1:x2]

                    color = detect_traffic_light_color_kmeans(crop)
                    cv2.putText(frame, color, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                                (0,0,255), 2)
      




    # vẽ vạch
    cv2.rectangle(frame, (c_line_x1,c_line_y1), (c_line_x2,c_line_y2), (0,255,0), 2)

    cv2.putText(frame, f"Count: {count}", (50,50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.imshow("Tracking", frame)

    if cv2.waitKey(1) == 27:
        break
cap.release()
cv2.destroyAllWindows()


0: 384x640 9 persons, 3 cars, 4 motorcycles, 4 traffic lights, 1 handbag, 300.2ms
Speed: 14.5ms preprocess, 300.2ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 persons, 3 cars, 4 motorcycles, 4 traffic lights, 1 handbag, 316.7ms
Speed: 6.7ms preprocess, 316.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 persons, 3 cars, 4 motorcycles, 4 traffic lights, 256.4ms
Speed: 2.9ms preprocess, 256.4ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 3 cars, 5 motorcycles, 4 traffic lights, 256.1ms
Speed: 2.8ms preprocess, 256.1ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 persons, 2 cars, 7 motorcycles, 4 traffic lights, 250.5ms
Speed: 2.9ms preprocess, 250.5ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 persons, 2 cars, 6 motorcycles, 3 traffic lights, 249.2ms
Speed: 2.9ms preprocess, 249.2ms inference, 1.1ms postp

In [4]:
def iou(box1, box2):
    # box: (x1, y1, x2, y2)

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    # diện tích phần giao
    inter = max(0, x2 - x1) * max(0, y2 - y1)

    # diện tích từng box
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    # diện tích hợp
    union = area1 + area2 - inter

    return inter / union if union > 0 else 0

In [ ]:
from ultralytics import YOLO
import cv2
model = YOLO("yolov8s.pt") 
cap = cv2.VideoCapture(r"D:\cdio3\video_den_do.mp4")
history = {}
count_his = set()
count = 0
den_x1=730
den_y1=30
den_x2=780
den_y2=180
line_y = 700  
traffic_light_color = "green"
while True:
    ret, frame = cap.read()
    if not ret:
        break
    results = model.track(frame, persist=True, tracker="custom_bytetrack.yaml")
    for r in results:
        boxes = r.boxes
        for box in boxes:
            if box.id is None:
                continue
            track_id = int(box.id[0])
            cls = int(box.cls[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            if cls == 9:
                overlap=iou((x1,y1,x2,y2),(den_x1,den_y1,den_x2,den_y2))
                if overlap>0.5:
                    crop = frame[y1:y2, x1:x2]
                    traffic_light_color = detect_traffic_light_color_kmeans(crop)
                # cv2.putText(frame, traffic_light_color, (x1, y1-10),
                #             cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
            if cls in [2, 3, 5, 7]:
                if track_id not in history:
                    history[track_id] = cy
                    continue
                prev_y = history[track_id]
                history[track_id] = cy
                # print(f"mau : {traffic_light_color}")
                if prev_y > line_y and cy <= line_y:
                    if track_id not in count_his:
                        if traffic_light_color.lower() == "red":

                            count += 1
                            count_his.add(track_id)
                            # print(f"VI PHAM ID: {track_id}")
                # cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
                # text=f"ID {track_id}"
                if track_id in count_his :
                    # text=f"đã đếm"
                    cv2.rectangle(frame, (x1,y1), (x2,y2), (0,0,255), 2)
                # cv2.putText(frame, text, (x1, y1-10),
                #             cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)
    # print(f"y{history[11]}")
    cv2.line(frame, (0, line_y), (frame.shape[1], line_y), (0,255,0), 2)
    # cv2.rectangle(frame, (den_x1,den_y1), (den_x2,den_y2), (0,255,0), 2)
    cv2.putText(frame, f"Count: {count}", (50,50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)
    cv2.imshow("Tracking", frame)
    if cv2.waitKey(1) == 27:
        break
cap.release()
cv2.destroyAllWindows()


0: 384x640 7 persons, 3 cars, 5 motorcycles, 4 traffic lights, 1 handbag, 357.2ms
Speed: 3.4ms preprocess, 357.2ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 2 cars, 4 motorcycles, 3 traffic lights, 1 handbag, 361.9ms
Speed: 46.5ms preprocess, 361.9ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 persons, 3 cars, 3 motorcycles, 2 traffic lights, 294.3ms
Speed: 8.3ms preprocess, 294.3ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 persons, 2 cars, 5 motorcycles, 2 traffic lights, 263.5ms
Speed: 7.0ms preprocess, 263.5ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 persons, 2 cars, 6 motorcycles, 1 traffic light, 265.5ms
Speed: 4.8ms preprocess, 265.5ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 persons, 1 car, 6 motorcycles, 2 traffic lights, 251.5ms
Speed: 6.2ms preprocess, 251.5ms inference, 1.5ms postpro

In [16]:
import ultralytics
import os

print(os.path.dirname(ultralytics.__file__))

c:\Users\HI\AppData\Local\Programs\Python\Python310\lib\site-packages\ultralytics
